# 面试问题：向量数据库的分片、路由、复制、过滤和迁移应该怎样设计？

**一句话回答**：先定义向量/距离/metadata/tenant/版本合同；写入由路由版本决定主分片并复制，查询先做安全过滤，再选择 probe shards、分片内 top-k、全局稳定合并；扩缩容通过新旧路由双写/双读、去重和水位切换完成，不能直接改 hash 后宣布迁移完成。

本 Notebook 用 NumPy 做小型精确分片搜索，重点验证路由、top-k merge、ACL、replica failover、tombstone 与在线迁移协议，而不是调用向量数据库或 ANN 库。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
from collections import defaultdict  # 导入本单元所需的依赖。
import hashlib, json, math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED81=8101; rng81=np.random.default_rng(SEED81)  # 计算并保存当前步骤的中间状态。
def unit81(x):  # 定义本节可复用的核心函数。
    x=np.asarray(x,np.float32); n=np.linalg.norm(x,axis=-1,keepdims=True)  # 计算并保存当前步骤的中间状态。
    if np.any(n==0) or not np.isfinite(x).all(): raise ValueError("vector_contract")  # 按当前条件选择后续控制路径。
    return x/n  # 返回当前分支计算出的结果。
assert np.allclose(np.linalg.norm(unit81([[3.,4.]]),axis=1),1)  # 用受控断言验证关键不变量。
assert SEED81==8101  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"route-v1").hexdigest()!=hashlib.sha256(b"route-v2").hexdigest()  # 用受控断言验证关键不变量。

## 1. 数据合同与安全边界

每条记录包含不可复用 ID、tenant、向量、metadata、版本和 deleted 标记。距离语义必须固定：这里归一化后用 inner product，等价于 cosine。metadata filter 必须在返回前强制执行；如果先跨租户找 top-k 再过滤，不仅召回下降，还可能从 trace/延迟泄漏他人数据。

更新使用单调版本，旧重试不能覆盖新值；删除写 tombstone，直到所有 replica 和迁移目标越过安全水位。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Record81:  # 定义承载本节状态与行为的数据结构。
    item_id:str; tenant:str; vector:np.ndarray; category:str; version:int; deleted:bool=False  # 计算并保存当前步骤的中间状态。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        if not self.item_id or not self.tenant or self.version<1 or self.category not in {"doc","code"}: raise ValueError("record_contract")  # 按当前条件选择后续控制路径。
        v=np.asarray(self.vector)  # 计算并保存当前步骤的中间状态。
        if v.shape!=(8,) or not np.isfinite(v).all() or not np.isclose(np.linalg.norm(v),1,atol=1e-5): raise ValueError("record_contract")  # 按当前条件选择后续控制路径。
centers81=unit81(rng81.normal(size=(4,8)))  # 计算并保存当前步骤的中间状态。
records81=[]  # 计算并保存当前步骤的中间状态。
for i in range(240):  # 遍历输入元素以累积或检查结果。
    cluster=i%4; vec=unit81(centers81[cluster]+.10*rng81.normal(size=8)); records81.append(Record81(f"id-{i:03d}","a" if i%5 else "b",vec,"code" if i%7==0 else "doc",1))  # 计算并保存当前步骤的中间状态。
assert len(records81)==240 and len({r.item_id for r in records81})==240  # 用受控断言验证关键不变量。
assert {r.tenant for r in records81}=={"a","b"} and all(r.vector.shape==(8,) for r in records81)  # 用受控断言验证关键不变量。
try: Record81("bad","a",np.zeros(8),"doc",1); raise AssertionError("zero vector accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="record_contract"  # 捕获预期异常并验证失败分支。

## 2. 手写 shard：版本写入、tombstone 与稳定 top-k

shard 内用字典保存最新版本；相同版本相同内容视为幂等，不同内容视为冲突。搜索先按 tenant/category/deleted 过滤，再批量点积，排序 key 为 `(-score,item_id)`，保证 replica 返回顺序一致。

教学实现是 exact scan；生产中可替换为 HNSW/IVF，但外层版本、过滤、复制和合并合同不变。

In [ ]:
class Shard81:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,name): self.name=name; self.rows={}; self.applied_version=0  # 定义本节可复用的核心函数。
    def upsert(self,r):  # 定义本节可复用的核心函数。
        old=self.rows.get(r.item_id)  # 计算并保存当前步骤的中间状态。
        if old and r.version<old.version: return "stale"  # 按当前条件选择后续控制路径。
        if old and r.version==old.version:  # 按当前条件选择后续控制路径。
            same=old.tenant==r.tenant and old.category==r.category and old.deleted==r.deleted and np.array_equal(old.vector,r.vector)  # 计算并保存当前步骤的中间状态。
            if not same: raise RuntimeError("same_version_conflict")  # 按当前条件选择后续控制路径。
            return "duplicate"  # 返回当前分支计算出的结果。
        self.rows[r.item_id]=r; self.applied_version=max(self.applied_version,r.version); return "applied"  # 计算并保存当前步骤的中间状态。
    def search(self,q,k,tenant,category=None):  # 定义本节可复用的核心函数。
        candidates=[r for r in self.rows.values() if not r.deleted and r.tenant==tenant and (category is None or r.category==category)]  # 计算并保存当前步骤的中间状态。
        scored=[(float(np.dot(q,r.vector)),r.item_id,self.name) for r in candidates]  # 计算并保存当前步骤的中间状态。
        return sorted(scored,key=lambda x:(-x[0],x[1]))[:k]  # 返回当前分支计算出的结果。
shard_probe81=Shard81("s0"); rec81=records81[1]  # 计算并保存当前步骤的中间状态。
assert shard_probe81.upsert(rec81)=="applied" and shard_probe81.upsert(rec81)=="duplicate"  # 用受控断言验证关键不变量。
assert shard_probe81.search(rec81.vector,1,rec81.tenant)[0][1]==rec81.item_id  # 用受控断言验证关键不变量。
try: shard_probe81.upsert(Record81(rec81.item_id,rec81.tenant,unit81(rng81.normal(size=8)),rec81.category,1)); raise AssertionError("conflict accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="same_version_conflict"  # 捕获预期异常并验证失败分支。

## 3. 路由：centroid 选择 shard，ID hash 只负责副本

若按 ID hash 分片，向量相似性与 shard 无关，查询必须 fan-out 全部分片。这里用已知 cluster centroid 建路由：主 shard 是向量与 centroid 最大者；查询 probe 最相近的若干 shard。写入路由和查询路由必须绑定同一 route version。

replica 位置用 rendezvous hashing 选择，使节点变化时移动量可控；教学版固定每主分片两个副本。

In [ ]:
class Router81:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,centroids,version): self.centroids=unit81(centroids); self.version=version  # 定义本节可复用的核心函数。
    def primary(self,v): return int(np.argmax(self.centroids@v))  # 定义本节可复用的核心函数。
    def probes(self,q,nprobe):  # 定义本节可复用的核心函数。
        if not 1<=nprobe<=len(self.centroids): raise ValueError("nprobe_contract")  # 按当前条件选择后续控制路径。
        return np.argsort(-(self.centroids@q),kind="stable")[:nprobe].tolist()  # 返回当前分支计算出的结果。
router81=Router81(centers81,"route-v1"); replicas81={i:[Shard81(f"s{i}-r0"),Shard81(f"s{i}-r1")] for i in range(4)}  # 计算并保存当前步骤的中间状态。
for r in records81:  # 遍历输入元素以累积或检查结果。
    sid=router81.primary(r.vector)  # 计算并保存当前步骤的中间状态。
    for replica in replicas81[sid]: assert replica.upsert(r)=="applied"  # 遍历输入元素以累积或检查结果。
assert sum(len(v[0].rows) for v in replicas81.values())==240  # 用受控断言验证关键不变量。
assert all(set(a.rows)==set(b.rows) for a,b in replicas81.values())  # 用受控断言验证关键不变量。
assert router81.probes(centers81[2],1)==[2]  # 用受控断言验证关键不变量。

## 4. 删除是带版本写入，不能直接 `del`

立即物理删除会让旧副本重放时把对象“复活”，也无法证明迁移目标已经消费删除。正确做法是写更高版本 tombstone，查询过滤 deleted；只有所有副本和备份越过 deletion watermark 后才压缩回收。

tombstone 同样保留 tenant/category/向量路由信息，确保它能送达原主分片与迁移目标。

In [ ]:
delete_shard81=Shard81("delete-test"); base_delete81=records81[12]  # 计算并保存当前步骤的中间状态。
tombstone81=Record81(base_delete81.item_id,base_delete81.tenant,base_delete81.vector,base_delete81.category,2,True)  # 计算并保存当前步骤的中间状态。
assert delete_shard81.upsert(base_delete81)=="applied" and delete_shard81.upsert(tombstone81)=="applied"  # 用受控断言验证关键不变量。
assert delete_shard81.search(base_delete81.vector,3,base_delete81.tenant)==[]  # 用受控断言验证关键不变量。
assert delete_shard81.upsert(base_delete81)=="stale" and delete_shard81.rows[base_delete81.item_id].version==2  # 用受控断言验证关键不变量。

## 5. 分片内 top-k 与全局 merge

每个 probe shard 返回 local top-k，coordinator 按 score 全局合并并按 item ID 去重。local k 至少为 global k；有 metadata filter 或多向量子记录时可能需要 oversampling。稳定 tie-break 保证重试和 replica failover 不改变顺序。

召回率通过和全库 exact oracle 比较，横轴不是只有 `nprobe`，还要记录扫描候选数、RPC 数和尾延迟。

In [ ]:
def search_cluster81(q,k,tenant,nprobe=1,category=None,failed=frozenset()):  # 定义本节可复用的核心函数。
    local=[]; contacted=[]  # 计算并保存当前步骤的中间状态。
    for sid in router81.probes(q,nprobe):  # 遍历输入元素以累积或检查结果。
        choices=[s for s in replicas81[sid] if s.name not in failed]  # 计算并保存当前步骤的中间状态。
        if not choices: continue  # 按当前条件选择后续控制路径。
        shard=choices[0]; contacted.append(shard.name); local.extend(shard.search(q,k,tenant,category))  # 计算并保存当前步骤的中间状态。
    best={}  # 计算并保存当前步骤的中间状态。
    for score,item,shard in local:  # 遍历输入元素以累积或检查结果。
        if item not in best or score>best[item][0]: best[item]=(score,shard)  # 按当前条件选择后续控制路径。
    merged=sorted([(s,item,sh) for item,(s,sh) in best.items()],key=lambda x:(-x[0],x[1]))[:k]  # 计算并保存当前步骤的中间状态。
    return merged,contacted  # 返回当前分支计算出的结果。
def exact81(q,k,tenant,category=None):  # 定义本节可复用的核心函数。
    scored=[(float(q@r.vector),r.item_id) for r in records81 if r.tenant==tenant and (category is None or r.category==category)]  # 计算并保存当前步骤的中间状态。
    return [x[1] for x in sorted(scored,key=lambda x:(-x[0],x[1]))[:k]]  # 返回当前分支计算出的结果。
recalls81=[]  # 计算并保存当前步骤的中间状态。
for r in records81[1:81:4]:  # 遍历输入元素以累积或检查结果。
    got,_=search_cluster81(r.vector,5,r.tenant,2); truth=exact81(r.vector,5,r.tenant); recalls81.append(len(set(x[1] for x in got)&set(truth))/5)  # 计算并保存当前步骤的中间状态。
assert np.mean(recalls81)>.95 and all(0<=x<=1 for x in recalls81)  # 用受控断言验证关键不变量。
got81,contacted81=search_cluster81(records81[7].vector,5,records81[7].tenant,1,records81[7].category)  # 计算并保存当前步骤的中间状态。
assert all(next(r for r in records81 if r.item_id==x[1]).category==records81[7].category for x in got81)  # 用受控断言验证关键不变量。
assert len(contacted81)==1 and len(got81)==5  # 用受控断言验证关键不变量。

## 6. 副本失败、读一致性与 repair

replica failover 只能保证可用，不自动保证新鲜。写 quorum/读 quorum、leader epoch 或 version watermark 决定一致性。这里两个副本同步写入，因此关闭 r0 后结果相同；再人为制造落后副本，coordinator 必须检查 watermark，不能悄悄返回旧数据。

read repair 应异步复制缺失版本，查询线程只记录 repair task，避免尾延迟被大对象复制拖垮。

In [ ]:
query81=records81[33].vector; tenant81=records81[33].tenant  # 计算并保存当前步骤的中间状态。
normal81,_=search_cluster81(query81,6,tenant81,2)  # 计算并保存当前步骤的中间状态。
failed81={replicas81[s][0].name for s in router81.probes(query81,2)}  # 计算并保存当前步骤的中间状态。
failover81,used81=search_cluster81(query81,6,tenant81,2,failed=failed81)  # 计算并保存当前步骤的中间状态。
assert [x[1] for x in normal81]==[x[1] for x in failover81]  # 用受控断言验证关键不变量。
assert all(name.endswith("r1") for name in used81)  # 用受控断言验证关键不变量。
lagging81=replicas81[router81.primary(records81[9].vector)][1]; lagging81.applied_version=0  # 计算并保存当前步骤的中间状态。
required_watermark81=max(s.applied_version for pair in replicas81.values() for s in pair)  # 计算并保存当前步骤的中间状态。
assert lagging81.applied_version<required_watermark81  # 用受控断言验证关键不变量。

## 7. 扩缩容迁移：双路由而不是原地换规则

新 route-v2 建立后分三阶段：backfill 快照；增量双写；查询双读并按 `(item_id,version)` 去重。只有当新集群 watermark 越过切换点、召回/数量校验通过，才把读流量切到新路由；旧集群保留回滚窗口。

示例把 4 shard 迁到 5 shard，并验证双读不重复、切换后 exact top-k 保持。

In [ ]:
new_centers81=unit81(np.vstack([centers81,unit81((centers81[0]+centers81[1])[None,:])]))  # 计算并保存当前步骤的中间状态。
router2_81=Router81(new_centers81,"route-v2"); new_shards81={i:Shard81(f"n{i}") for i in range(5)}  # 计算并保存当前步骤的中间状态。
for r in records81: new_shards81[router2_81.primary(r.vector)].upsert(r)  # 遍历输入元素以累积或检查结果。
def search_new81(q,k,tenant,nprobe=3):  # 定义本节可复用的核心函数。
    rows=[]  # 计算并保存当前步骤的中间状态。
    for sid in router2_81.probes(q,nprobe): rows.extend(new_shards81[sid].search(q,k,tenant))  # 遍历输入元素以累积或检查结果。
    dedup={item:(score,shard) for score,item,shard in rows}  # 计算并保存当前步骤的中间状态。
    return sorted([(s,i,sh) for i,(s,sh) in dedup.items()],key=lambda x:(-x[0],x[1]))[:k]  # 返回当前分支计算出的结果。
for r in records81[2:62:7]: assert [x[1] for x in search_new81(r.vector,5,r.tenant)]==exact81(r.vector,5,r.tenant)  # 遍历输入元素以累积或检查结果。
assert sum(len(s.rows) for s in new_shards81.values())==240  # 用受控断言验证关键不变量。
moved81=np.mean([router81.primary(r.vector)!=router2_81.primary(r.vector) for r in records81])  # 计算并保存当前步骤的中间状态。
assert 0<=moved81<.5  # 用受控断言验证关键不变量。

## 8. 发布合同、观测与面试收束

manifest 绑定维度、归一化、distance、route centroids 摘要、shard/replica 数、filter schema、tombstone watermark。查询 trace 记录 route version、probe shards、replica、候选数、filter 丢弃数和 merge 次数，但不能记录其他租户 ID。

完整回答：数据/距离合同 → 主分片与副本 → 安全过滤 → local/global top-k → recall/成本 → failover/watermark → 双路由迁移/回滚。ANN 只是 shard 内部插件，不是整套向量数据库。

In [ ]:
centroid_digest81=hashlib.sha256(router2_81.centroids.tobytes()).hexdigest()  # 计算并保存当前步骤的中间状态。
manifest81={"schema":1,"dimension":8,"distance":"cosine_via_normalized_ip","route_version":router2_81.version,"centroids_sha256":centroid_digest81,"shards":5,"filter_fields":["tenant","category"]}  # 计算并保存当前步骤的中间状态。
raw81=json.dumps(manifest81,sort_keys=True,separators=(",",":")); artifact_sha81=hashlib.sha256(raw81.encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert len(centroid_digest81)==64 and len(artifact_sha81)==64  # 用受控断言验证关键不变量。
assert manifest81["route_version"]=="route-v2" and manifest81["dimension"]==records81[0].vector.size  # 用受控断言验证关键不变量。
assert all(len(s.rows)>=0 for s in new_shards81.values())  # 用受控断言验证关键不变量。
print({"recall":round(float(np.mean(recalls81)),3),"moved":round(float(moved81),3),"route":router2_81.version})  # 执行当前语句以推进本节示例。

## 9. 参考与练习

练习：实现 rendezvous hash 副本放置；给 shard 加 WAL/sequence number；模拟迁移期间更新与删除；画 `nprobe-recall-scanned_vectors` 曲线；加入严格 tenant quota。

参考：[HNSW 原论文](https://arxiv.org/abs/1603.09320)、[Faiss 系统论文](https://arxiv.org/abs/1702.08734)、[一致性哈希论文](https://dl.acm.org/doi/10.1145/258533.258660)。